In [1]:
import evdev
import time
import select

dev = evdev.InputDevice("/dev/input/event9")

In [2]:
# List all input devices
devices = [evdev.InputDevice(path) for path in evdev.list_devices()]
for d in devices:
    print(d.path, d.name, d.phys)

# Pick the 8BitDo controller (adjust index/name match as needed)
dev = next((d for d in devices if "8BitDo" in d.name), None)
if not dev:
    raise SystemExit("Controller not found")

/dev/input/event11 8BitDo 8BitDo Receiver Mouse usb-xhci-hcd.0-1.4/input1
/dev/input/event10 8BitDo 8BitDo Receiver Keyboard usb-xhci-hcd.0-1.4/input1
/dev/input/event9 8BitDo Ultimate Wireless / Pro 2 Wired Controller usb-xhci-hcd.0-1.4/input0
/dev/input/event8 vc4-hdmi-1 HDMI Jack ALSA
/dev/input/event7 vc4-hdmi-1 vc4-hdmi-1/input0
/dev/input/event6 vc4-hdmi-0 HDMI Jack ALSA
/dev/input/event5 vc4-hdmi-0 vc4-hdmi-0/input0
/dev/input/event4 pwr_button gpio-keys/input0
/dev/input/event3 Logitech USB Keyboard System Control usb-xhci-hcd.0-2/input1
/dev/input/event2 Logitech USB Keyboard Consumer Control usb-xhci-hcd.0-2/input1
/dev/input/event1 Logitech USB Keyboard usb-xhci-hcd.0-2/input0
/dev/input/event0 USB Optical Mouse usb-xhci-hcd.1-2/input0


In [3]:
# Current controller state
state = {
    "lx": 0, "ly": 0,   # left stick
    "rx": 0, "ry": 0,   # right stick
    "buttons": set(),
}

ABS_MAP = {
    evdev.ecodes.ABS_X: "lx",
    evdev.ecodes.ABS_Y: "ly",
    evdev.ecodes.ABS_RX: "rx",
    evdev.ecodes.ABS_RY: "ry",
}

In [4]:
def drain_events():
    """Non-blocking: read all pending events, update state, return immediately."""
    r, _, _ = select.select([dev.fd], [], [], 0)
    if not r:
        return
    for event in dev.read():
        if event.type == evdev.ecodes.EV_ABS and event.code in ABS_MAP:
            state[ABS_MAP[event.code]] = event.value
        elif event.type == evdev.ecodes.EV_KEY:
            if event.value == 1:
                state["buttons"].add(event.code)
            elif event.value == 0:
                state["buttons"].discard(event.code)

In [5]:
dev = evdev.InputDevice("/dev/input/event9")  # or updated path
caps = dev.capabilities(verbose=False)
print(caps.keys())

dict_keys([0, 1, 3, 21])


In [6]:
caps = dev.capabilities(verbose=False)
for code in [evdev.ecodes.ABS_X, evdev.ecodes.ABS_Y, evdev.ecodes.ABS_RX, evdev.ecodes.ABS_RY]:
    absinfo = dev.absinfo(code)
    print(code, absinfo)

0 value 0, min -32768, max 32767, fuzz 16, flat 128, res 0
1 value -1, min -32768, max 32767, fuzz 16, flat 128, res 0
3 value 0, min -32768, max 32767, fuzz 16, flat 128, res 0
4 value 0, min -32768, max 32767, fuzz 16, flat 128, res 0


In [7]:
CENTER = {
    "lx": -1792, "ly": 3327,
    "rx": -4096, "ry": 1023,
}
AXIS_MIN, AXIS_MAX = -32768, 32767

def normalize(value, center, deadzone=1500):
    v = value - center
    if abs(v) < deadzone:
        return 0.0
    if v < 0:
        return v / (center - AXIS_MIN)
    else:
        return v / (AXIS_MAX - center)

def normalized_state(state):
    return {
        "lx": normalize(-state["lx"], CENTER["lx"]),
        "ly": normalize(-state["ly"], CENTER["ly"]),
        "rx": normalize(-state["rx"], CENTER["rx"]),
        "ry": normalize(-state["ry"], CENTER["ry"]),
        "buttons": state["buttons"],
    }

In [8]:
caps = dev.capabilities(verbose=False)
print(caps.get(evdev.ecodes.EV_ABS))

[(0, AbsInfo(value=0, min=-32768, max=32767, fuzz=16, flat=128, resolution=0)), (1, AbsInfo(value=-1, min=-32768, max=32767, fuzz=16, flat=128, resolution=0)), (2, AbsInfo(value=0, min=0, max=255, fuzz=0, flat=0, resolution=0)), (3, AbsInfo(value=0, min=-32768, max=32767, fuzz=16, flat=128, resolution=0)), (4, AbsInfo(value=0, min=-32768, max=32767, fuzz=16, flat=128, resolution=0)), (5, AbsInfo(value=0, min=0, max=255, fuzz=0, flat=0, resolution=0)), (16, AbsInfo(value=0, min=-1, max=1, fuzz=0, flat=0, resolution=0)), (17, AbsInfo(value=0, min=-1, max=1, fuzz=0, flat=0, resolution=0))]


In [9]:
def arcade_drive(throttle, steering):
    """
    throttle: -1.0 (full reverse) to 1.0 (full forward)
    steering: -1.0 (full left) to 1.0 (full right)
    returns (left, right) each -1.0 to 1.0
    """
    left = throttle + steering
    right = throttle - steering

    # normalize back into range if combined value exceeds ±1.0
    max_mag = max(abs(left), abs(right), 1.0)
    left /= max_mag
    right /= max_mag

    return left, right

In [12]:
import serial
import struct
import time

ser = serial.Serial('/dev/ttyACM1', 115200, timeout=1)  # adjust port as needed

# Matches TPBot.TrackingState in elecfreaks/pxt-TPBot (V1.ts / V2.ts):
#   L_R_line          = 0  -> both sensors on line (both Black)
#   L_unline_R_line    = 1  -> left off line, right on line
#   L_line_R_unline    = 2  -> left on line, right off line
#   L_R_unline         = 3  -> both sensors off line (both White)
TRACKING_STATES = {
    0: "L_R_line",
    1: "L_unline_R_line",
    2: "L_line_R_unline",
    3: "L_R_unline",
}

def send_motor_command(left_pct, right_pct):
    left_pct = max(-100, min(100, left_pct))
    right_pct = max(-100, min(100, right_pct))
    robot_pct = 1
    command_pct = 1
    
    line = f"{robot_pct},{command_pct},{left_pct},{right_pct}\n"
    ser.write(line.encode('ascii'))

    # The base station firmware may emit '#'-prefixed debug lines before
    # the real reply - skip those and take the first real line.
    for _ in range(10):
        reply = ser.readline().decode('ascii', errors='ignore').strip()
        if not reply:
            return None
        if reply.startswith('#'):
            continue
        try:
            code = int(reply)
            return TRACKING_STATES.get(code, f"UNKNOWN({code})")
        except ValueError:
            return None
    return None

HZ = 2
INTERVAL = 1.0 / HZ

while True:
    t0 = time.time()
    drain_events()
    ns = normalized_state(state)
    left, right = arcade_drive(throttle=ns["ly"], steering=ns["lx"])
    left_pct = int(left * 100)
    right_pct = int(right * 100)
    #tracking = send_motor_command(left_pct, right_pct)
    tracking = send_motor_command(right_pct,left_pct)
    print("left motor:", left_pct, "right_motor:", right_pct, "tracking:", tracking)
    elapsed = time.time() - t0
    time.sleep(max(0, INTERVAL - elapsed))


left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: 52 right_motor: 34 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: -45 right_motor: 100 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: -1 right_motor: -17 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: 63 right_motor: -100 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: 100 right_motor: 100 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: 43 right_motor: -100 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L_R_unline
left motor: -4 right_motor: -15 tracking: L

KeyboardInterrupt: 